In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# 1. 定义数据仓库 (Custom Dataset)s
class LinearDataset(Dataset):
    def __init__(self, num_samples=1000):
        # 构造 1000 条 3 维特征的数据
        self.X = torch.randn(num_samples, 3)
        true_w = torch.tensor([[2.0], [-1.0], [0.5]])
        true_b = 0.5
        self.y = torch.matmul(self.X, true_w) + true_b + 0.1 * torch.randn(num_samples, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # 就像点名一样，根据索引返回对应的特征和标签
        return self.X[idx], self.y[idx]

# 2. 定义模型 (nn.Module)
class LinearModel(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.linear = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.linear(x)

# ----------------- 准备工作 -----------------

# A. 实例化数据并拆分 (训练集 80%, 验证集 20%)
full_dataset = LinearDataset(num_samples=1000)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size])

# B. 实例化 DataLoader (快递车)
# 训练集需要打乱 (shuffle)，验证集通常不需要
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

# C. 模型、损失函数、优化器
model = LinearModel(in_features=3)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# ----------------- 核心训练循环 -----------------

epochs = 50
for epoch in range(epochs):
    # --- 【训练阶段】 ---
    model.train()  # 切换到训练模式
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        # 1. 前向传播
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # 2. 三步走：清零 -> 反向传播 -> 更新
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 累加 Batch 损失 (用于后面算平均值)
        train_loss += loss.item()

    # --- 【验证阶段】 ---
    model.eval()  # 切换到评估模式
    val_loss = 0.0
    with torch.no_grad():  # 彻底关闭梯度计算，节省显存和计算资源
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()

    # 计算平均 Loss 并打印
    if (epoch + 1) % 5 == 0:
        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        print(f"Epoch [{epoch+1:>2}/{epochs}] | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

print("\n训练与验证完成！")